# NB10 — Q2 — Are the three dials the same thing?

        **CPU only — turn the accelerator OFF. ~5 minutes. One account.**


> **New here?** Read `05_PLAIN_ENGLISH_GUIDE.md` first — it explains what this
> project is measuring and why, without jargon. This notebook assumes you have.


        ## The question

        > If an image needs more depth, does it also need more resolution and more numeric precision?

        ## In plain English

        We can reduce compute three ways: fewer layers, smaller images, fewer bits. We measured all three on the same models and the same images. Now we ask whether they're measuring one underlying thing or three separate things.

        ## Why it matters

        **Nobody has ever asked this.** Every adaptive-inference paper picks one dial — nearly always depth — and treats it as *the* compute axis. If one shared factor explains most of the variation, that assumption is validated and a single compute-need number is justified. If it doesn't, then results about early-exit depth say nothing about precision-adaptive inference, and a lot of published generalisation is unwarranted. Either answer is a contribution, and the data comes free once the atlas exists — the best novelty-per-GPU-hour in the project.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   4128d47f9964   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  6abdba4ff104   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

## Step 1 — Load

In [ ]:
ACCOUNT = 'acct1'      # <<< CHANGE ME

sess = msc.Session(account=ACCOUNT, phase='analysis', dataset='cifar100',
                   enable_hf=True)
# Metrics and per-sample tables only -- checkpoints excluded. Fast.
sess.sync_state(include_checkpoints=False, verbose=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt

# Inventory: what do we actually have to work with?
runs = {}
for d in sorted(sess.runs_dir.iterdir()) if sess.runs_dir.exists() else []:
    ps = d / 'per_sample'
    m = msc.read_json(ps / 'meta.json', default=None)
    if m and (ps / 'test.parquet').exists():
        runs[d.name] = m
budgets = {r: sess.budgets(m['arch']) for r, m in runs.items()}

inv = pd.DataFrame([{'run_id': k, 'arch': v['arch'], 'family': v['family'],
                     'seed': v['seed'],
                     'order': v['sample_order_hash'][:10]} for k, v in runs.items()])
if len(inv):
    inv = inv.sort_values(['family', 'arch', 'seed'])
    display(inv)
    print(f"\n{len(runs)} measured models   "
          f"{inv.order.nunique()} distinct image orderings (must be 1)")
else:
    # Not an error -- it means the measurement notebook has not run yet on any
    # model this account can see.
    trained = [d.name for d in sorted(sess.runs_dir.iterdir())
               if (d / 'summary.json').exists()] if sess.runs_dir.exists() else []
    print('No measured models found.')
    print()
    if trained:
        print(f'{len(trained)} run(s) have finished TRAINING but not MEASUREMENT:')
        for t in trained[:10]:
            print(f'  {t}')
        print()
        print('-> Run NB02 (Phase 0) or NB08 (atlas) to produce the per-sample')
        print('   tables, then re-run this notebook.')
    else:
        print('-> No completed runs at all. Run sess.sync_state(), or finish')
        print('   the training notebooks first.')

## Step 2 — Principal component analysis across the dials

`pc1_variance` is the fraction of variation explained by a single shared
factor. Our pre-registered prediction (H2) is ≥ 0.60.

**Which resolution measurement:** we use `res_proxy` (shrink-then-restore)
as the primary, because it is defined for **all 15 architectures**.
MLP-Mixer cannot run at another resolution at all — its token-mixing
layer is a linear map whose input dimension is the patch count — so
making native primary would mean measuring one architecture differently
from the other fourteen, and any cross-architecture claim on this axis
would then compare two different quantities.

Native is reported as a robustness check in Step 5, for the 14 that
support it.

In [ ]:
q2_all = []
for r, m in runs.items():
    if m['seed'] != 1:
        continue                     # one seed per architecture is enough here
    try:
        d = msc.analyse_q2_axis_structure(sess.data_dir, r, budgets[r],
                                          axes=('depth', 'res_proxy', 'precision'))
        d['arch'] = m['arch']; d['family'] = m['family']
        q2_all.append(d)
    except Exception as e:
        print(f'  {r}: {e}')
q2 = pd.concat(q2_all, ignore_index=True) if q2_all else pd.DataFrame()
if not len(q2):
    print('No axis structure computed -- no measured runs found (run NB08).')
if len(q2):
    msc.save_analysis(sess.data_dir, 'q2_axis_structure_all', q2, sess.hub)
    display(q2.pivot_table(index='arch', columns='tau',
                           values='pc1_variance').round(3))
    frac = (q2.pc1_variance >= 0.6).mean()
    print(f'\nH2 predicts PC1 >= 0.60.')
    print(f'Cells clearing it: {frac:.0%}')
    print('\n  Mostly above  -> one shared "compute need". Single router justified.')
    print('  Mostly below  -> the dials are different. Depth-only results do not')
    print('                   generalise, and that is a finding worth reporting.')

## Step 3 — Which dials agree with which?

Pairwise correlations. If depth and resolution correlate strongly but
precision doesn't, that's a more interesting story than a single number.

In [ ]:
if len(q2):
    cols = [c for c in q2.columns if c.startswith('rho_')]
    if cols:
        display(q2[q2.tau == 0.1][['arch'] + cols].round(3))
        long = q2[q2.tau == 0.1][cols].melt(var_name='pair', value_name='rho')
        display(long.groupby('pair').rho.agg(['mean', 'std', 'min', 'max']).round(3))

## Step 4 — Plot

In [ ]:
if len(q2):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for arch, g in q2.groupby('arch'):
        ax[0].plot(g.tau, g.pc1_variance, 'o-', label=arch)
    ax[0].axhline(0.6, ls='--', c='k', lw=1, label='H2 threshold')
    ax[0].set_xlabel('tau'); ax[0].set_ylabel('variance explained by PC1')
    ax[0].set_title('Q2: is compute-need one-dimensional?')
    ax[0].legend(fontsize=7, ncol=2); ax[0].grid(alpha=.3)

    load_cols = [c for c in q2.columns if c.startswith('loading_')]
    if load_cols:
        sub = q2[q2.tau == 0.1].set_index('arch')[load_cols]
        sub.plot(kind='bar', ax=ax[1])
        ax[1].set_ylabel('PC1 loading'); ax[1].set_title('How each dial loads on PC1')
        ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)
    plt.tight_layout()
    msc.save_figure(fig, sess.data_dir, 'q2_axis_structure', sess.hub)
    plt.show()

## Step 5 — Robustness check: native resolution vs the proxy

For the 14 architectures that *can* run at a smaller input, we measured
the resolution axis both ways. If the two agree, then using the proxy
uniformly (Step 2) costs us nothing, and the methodological caveat is
something we **measured** rather than merely argued about.

The `native_supported` column records which architectures could do both.
MLP-Mixer will show `False` — that's the documented limitation, not a
failure.

In [ ]:
rows = []
core = msc._import_msc_core()
for r, m in runs.items():
    if m['seed'] != 1:
        continue
    df = msc.load_per_sample(sess.data_dir, r)
    axes_here = msc.available_axes(df)
    native_ok = 'res_native' in axes_here
    rec = {'arch': m['arch'], 'native_supported': native_ok,
           'axes_available': ' '.join(axes_here)}
    if native_ok:
        for t in (0.0, 0.1, 0.3):
            a = msc.msc_for_run(df, budgets[r], 'res_native', t).clean()
            b = msc.msc_for_run(df, budgets[r], 'res_proxy', t).clean()
            rec[f'agree_tau{t}'] = core.spearman(a, b)
            if t == 0.1:
                rec['mean_native'] = float(np.nanmean(a))
                rec['mean_proxy'] = float(np.nanmean(b))
    rows.append(rec)
rp = pd.DataFrame(rows)
msc.save_analysis(sess.data_dir, 'q2_resolution_native_vs_proxy', rp, sess.hub)
display(rp.round(3))

ok = rp[rp.native_supported]
if len(ok) and 'agree_tau0.1' in ok:
    m_ = ok['agree_tau0.1'].median()
    print(f'\nMedian native-vs-proxy agreement: {m_:.3f} '
          f'across {len(ok)} architectures')
    if m_ > 0.9:
        print('  -> The two are near-interchangeable. Using the proxy uniformly')
        print('     is well justified, and we can say so with a number.')
    else:
        print('  -> They differ materially. Report BOTH in the paper and discuss;')
        print('     do not present either as if it were the other.')
n_no = int((~rp.native_supported).sum())
if n_no:
    print(f'\n{n_no} architecture(s) cannot run at native resolution by')
    print('construction. Stated as a limitation in the model card.')

## Step 6 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()

# D-19: draining the upload queue is NOT the same as the files being on
# HuggingFace, and "[SESSION] done" reads like a confirmation it is not.
# Ask the repository before you close this tab.
#
# D-20: three states, not two. FINISHED and RESUMABLE are both safe -- a run
# paused at epoch 120 whose ckpt_last.pt is on HF loses nothing when you close
# the tab. Only AT RISK (no summary.json AND no checkpoint) needs action.
try:
    _ids = [c['run_id'] for c in cfgs]
except NameError:
    _ids = []
if _ids:
    sess.confirm_on_hf(_ids)